In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss
import numpy as np

In [ ]:
def add_feature_engineering(df):
    df = df.copy()
    df["is_red_zone"] = (df["yardline_100"] <= 20).astype(int)
    df["is_goal_to_go"] = df["goal_to_go"].astype(int)

    df["distance_bucket"] = pd.cut(
        df["ydstogo"],
        bins=[0, 3, 6, 10, 20, 100],
        labels=["short", "medium", "long", "very_long", "desperation"])

    df["time_bucket"] = pd.cut(
        df["quarter_seconds_remaining"],
        bins=[0, 60, 300, 900],
        labels=["final_minute", "late_quarter", "early_quarter"])

    df["half"] = (df["qtr"] > 2).astype(int)

    df["is_trailing"] = (df["score_differential"] < 0).astype(int)
    df["is_blown_out"] = (abs(df["score_differential"]) >= 14).astype(int)


    df["down_distance"] = df["down"].astype(int) * df["ydstogo"]
    #rolling stats
    if "posteam" in df.columns and "epa" in df.columns:
        df = df.sort_values(["game_id", "drive", "play_id"]).reset_index(drop=True)
        df["rolling_epa_team"] = df.groupby("posteam")["epa"].transform(
            lambda x: x.rolling(5, min_periods=1).mean())

    for col in ["distance_bucket", "time_bucket"]:
        df[col] = df[col].astype("category")

    return df


In [ ]:
df = pd.read_csv("coverage_model_dataset.csv")
df = df.dropna()
df = df[df['down']!='4.0']
df = df[(df['play_type'] == 'pass') | (df['play_type'] == 'run')]
df.head()
df = add_feature_engineering(df)

In [ ]:
df['play_type'].value_counts()

,count
play_type,
pass,141368
run,100572


**Run or Pass Model**

In [ ]:
target = 'play_type'
features = [
    'posteam','defteam','yardline_100','quarter_seconds_remaining',
    'half_seconds_remaining','qtr','down','ydstogo','score_differential',
    'shotgun','no_huddle','is_red_zone','distance_bucket','time_bucket',
    'is_trailing','is_blown_out','half','down_distance'
]
X = df[features]
y = df[target]
categorical_cols = ['posteam','defteam','distance_bucket','time_bucket']

for col in categorical_cols:
    X[col] = X[col].astype("category")

/tmp/ipython-input-1276826475.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X[col] = X[col].astype("category")
/tmp/ipython-input-1276826475.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X[col] = X[col].astype("category")


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = LGBMClassifier(
    n_estimators=600,
    learning_rate=0.03,
    num_leaves=63,
    colsample_bytree=0.8,
    subsample=0.8)

model.fit(X_train, y_train, categorical_feature=categorical_cols)

[LightGBM] [Info] Number of positive: 80347, number of negative: 113205
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.019948 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 925
[LightGBM] [Info] Number of data points in the train set: 193552, number of used features: 18
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.415118 -> initscore=-0.342846
[LightGBM] [Info] Start training from score -0.342846


LGBMClassifier(colsample_bytree=0.8, learning_rate=0.03, n_estimators=600,
               num_leaves=63, subsample=0.8)

In [ ]:
preds = model.predict(X_test)
print(accuracy_score(y_test, preds))
print(classification_report(y_test, preds))

0.7328263205753492
              precision    recall  f1-score   support

        pass       0.76      0.79      0.77     28163
         run       0.69      0.65      0.67     20225

    accuracy                           0.73     48388
   macro avg       0.73      0.72      0.72     48388
weighted avg       0.73      0.73      0.73     48388



**First Down Probability Model**

In [ ]:
df = pd.read_csv("coverage_model_dataset_2.csv")
df = df.dropna()
df = df[df['down']!='4.0']
df = df[(df['play_type'] == 'pass') | (df['play_type'] == 'run')]
df.head()

,posteam,defteam,yardline_100,quarter_seconds_remaining,half_seconds_remaining,drive,sp,qtr,down,goal_to_go,...,xpass,first_down,complete_pass,air_yards,pass_length,pass_location,qb_scramble,qb_hit,xyac_mean_yardage,xyac_success
3,ATL,PHI,80.0,900.0,1800.0,1.0,0,1,1.0,0,...,0.587117,0.0,1.0,8.0,short,right,0,0.0,3.515878,0.998706
6,ATL,PHI,39.0,790.0,1690.0,1.0,0,1,1.0,0,...,0.457959,0.0,0.0,4.0,short,right,0,0.0,4.103093,0.611379
7,ATL,PHI,39.0,785.0,1685.0,1.0,0,1,2.0,0,...,0.511643,0.0,0.0,-3.0,short,left,0,0.0,9.786461,0.481282
8,ATL,PHI,39.0,781.0,1681.0,1.0,0,1,3.0,0,...,0.972056,1.0,1.0,24.0,deep,left,0,0.0,5.718963,1.000000
14,PHI,ATL,96.0,610.0,1510.0,2.0,0,1,2.0,0,...,0.587902,0.0,1.0,4.0,short,left,0,0.0,4.152075,0.622114


In [ ]:
df = add_feature_engineering(df)

In [ ]:
features_fd = [
    "yardline_100","down","ydstogo","qtr","quarter_seconds_remaining",
    "half_seconds_remaining","score_differential","posteam","defteam",
    "shotgun","no_huddle","qb_dropback","wp",
    "is_red_zone","distance_bucket","is_trailing","is_blown_out",
    "time_bucket","down_distance","half"]

In [ ]:
target = "first_down"
X = df[features_fd].copy()
y = df[target]
# categorical
cat_fd = ["posteam","defteam","distance_bucket","time_bucket"]

for c in cat_fd:
    X[c] = X[c].astype("category")
# split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model_fd = LGBMClassifier(
    n_estimators=700,
    learning_rate=0.03,
    num_leaves=63,
    subsample=0.8,
    colsample_bytree=0.8)

model_fd.fit(X_train, y_train, categorical_feature=cat_fd)
prob = model_fd.predict_proba(X_test)[:,1]

[LightGBM] [Info] Number of positive: 34568, number of negative: 60524
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.049144 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1172
[LightGBM] [Info] Number of data points in the train set: 95092, number of used features: 19
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.363522 -> initscore=-0.560112
[LightGBM] [Info] Start training from score -0.560112


In [ ]:
print("AUC:", roc_auc_score(y_test, prob))
print("Accuracy:", accuracy_score(y_test, model_fd.predict(X_test)))
print(classification_report(y_test, model_fd.predict(X_test)))
logloss = log_loss(y_test, prob)
print("Log Loss:", logloss)

AUC: 0.6581555579239319
Accuracy: 0.6643671391915198
              precision    recall  f1-score   support

         0.0       0.69      0.88      0.77     15231
         1.0       0.56      0.29      0.38      8542

    accuracy                           0.66     23773
   macro avg       0.63      0.58      0.58     23773
weighted avg       0.64      0.66      0.63     23773

Log Loss: 0.6162712492868192


**Completion Probability Model**

In [ ]:
df_xc = pd.read_csv("coverage_model_dataset_2.csv")
df_xc = df_xc.dropna()
df_xc = df_xc[df_xc['down']!='4.0']
df_xc = df_xc[(df_xc['play_type'] == 'pass')]
df_xc.head()

,posteam,defteam,yardline_100,quarter_seconds_remaining,half_seconds_remaining,drive,sp,qtr,down,goal_to_go,...,xpass,first_down,complete_pass,air_yards,pass_length,pass_location,qb_scramble,qb_hit,xyac_mean_yardage,xyac_success
3,ATL,PHI,80.0,900.0,1800.0,1.0,0,1,1.0,0,...,0.587117,0.0,1.0,8.0,short,right,0,0.0,3.515878,0.998706
6,ATL,PHI,39.0,790.0,1690.0,1.0,0,1,1.0,0,...,0.457959,0.0,0.0,4.0,short,right,0,0.0,4.103093,0.611379
7,ATL,PHI,39.0,785.0,1685.0,1.0,0,1,2.0,0,...,0.511643,0.0,0.0,-3.0,short,left,0,0.0,9.786461,0.481282
8,ATL,PHI,39.0,781.0,1681.0,1.0,0,1,3.0,0,...,0.972056,1.0,1.0,24.0,deep,left,0,0.0,5.718963,1.000000
14,PHI,ATL,96.0,610.0,1510.0,2.0,0,1,2.0,0,...,0.587902,0.0,1.0,4.0,short,left,0,0.0,4.152075,0.622114


In [ ]:
df_xc = add_feature_engineering(df_xc)

In [ ]:
df_xc.head()

,posteam,defteam,yardline_100,quarter_seconds_remaining,half_seconds_remaining,drive,sp,qtr,down,goal_to_go,...,xyac_mean_yardage,xyac_success,is_red_zone,is_goal_to_go,distance_bucket,time_bucket,half,is_trailing,is_blown_out,down_distance
3,ATL,PHI,80.0,900.0,1800.0,1.0,0,1,1.0,0,...,3.515878,0.998706,0,0,very_long,early_quarter,0,0,0,15
6,ATL,PHI,39.0,790.0,1690.0,1.0,0,1,1.0,0,...,4.103093,0.611379,0,0,long,early_quarter,0,0,0,10
7,ATL,PHI,39.0,785.0,1685.0,1.0,0,1,2.0,0,...,9.786461,0.481282,0,0,long,early_quarter,0,0,0,20
8,ATL,PHI,39.0,781.0,1681.0,1.0,0,1,3.0,0,...,5.718963,1.000000,0,0,long,early_quarter,0,0,0,30
14,PHI,ATL,96.0,610.0,1510.0,2.0,0,1,2.0,0,...,4.152075,0.622114,0,0,long,early_quarter,0,0,0,16


In [ ]:
features_xc = [
    "air_yards","pass_length","pass_location",
    "qb_dropback","qb_scramble","qb_hit",
    "yardline_100","down","ydstogo","qtr",
    "quarter_seconds_remaining","half_seconds_remaining",
    "score_differential","shotgun","no_huddle",
    "posteam","defteam",
    "is_red_zone","distance_bucket","is_trailing",
    "time_bucket","down_distance"]


In [ ]:
categorical_cols = ["posteam", "defteam", "pass_length", "pass_location", "qtr"]

In [ ]:
print(df_xc.columns.tolist())

['posteam', 'defteam', 'yardline_100', 'quarter_seconds_remaining', 'half_seconds_remaining', 'drive', 'sp', 'qtr', 'down', 'goal_to_go', 'ydstogo', 'ydsnet', 'shotgun', 'no_huddle', 'play_type', 'score_differential', 'qb_dropback', 'wp', 'xpass', 'first_down', 'complete_pass', 'air_yards', 'pass_length', 'pass_location', 'qb_scramble', 'qb_hit', 'xyac_mean_yardage', 'xyac_success', 'is_red_zone', 'is_goal_to_go', 'distance_bucket', 'time_bucket', 'half', 'is_trailing', 'is_blown_out', 'down_distance']


In [ ]:
target_xc = "complete_pass"

X = df_xc[features_xc].copy()
y = df_xc[target_xc]

cat_xc = ["posteam","defteam","pass_length","pass_location","distance_bucket","time_bucket"]

for c in cat_xc:
    X[c] = X[c].astype("category")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


model_xc = LGBMClassifier(
    n_estimators=900,
    learning_rate=0.02,
    num_leaves=127,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42)

model_xc.fit(X_train, y_train, categorical_feature=cat_xc)

prob = model_xc.predict_proba(X_test)[:,1]

[LightGBM] [Info] Number of positive: 65795, number of negative: 29297
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.027530 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 996
[LightGBM] [Info] Number of data points in the train set: 95092, number of used features: 20
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.691909 -> initscore=0.809059
[LightGBM] [Info] Start training from score 0.809059


In [ ]:
print("AUC:", roc_auc_score(y_test, prob))
print("Accuracy:", accuracy_score(y_test, model_xc.predict(X_test)))
print(classification_report(y_test, model_xc.predict(X_test)))
logloss = log_loss(y_test, prob)
print("Log Loss:", logloss)

AUC: 0.693663703918294
Accuracy: 0.7231312833887183
              precision    recall  f1-score   support

         0.0       0.58      0.28      0.38      7153
         1.0       0.75      0.91      0.82     16620

    accuracy                           0.72     23773
   macro avg       0.66      0.60      0.60     23773
weighted avg       0.70      0.72      0.69     23773

Log Loss: 0.5612713152421337
